# 01 데이터 수집: 회사 주변 식당 목록

카카오 로컬 API로 회사(여의도역 인근) 주변 식당을 모읍니다.

| 항목 | 내용 |
|---|---|
| 입력 | 카카오 REST API 키 (Colab 보안 비밀 `KAKAO_REST_API_KEY`) |
| 출력 | `restaurants.csv` (식당 1곳 = 1행, `data_type = 실제`) |
| 검증 | 중복 제거, 반경 밖 식당 제외, 카테고리 빈칸 확인 |

**실행 방법:** 위쪽 메뉴 **런타임 → 모두 실행**. 설정은 바로 아래 칸에서만 바꿉니다.

In [ ]:
# ── 설정 (여기만 바꾸세요) ──────────────────────────
COMPANY_ADDRESS = None   # 거리 계산 기준. 상세 주소는 파일에 적지 않고 Colab 보안 비밀 COMPANY_ADDRESS에 넣어요(여의도역 인근)
RADIUS_M = 1000          # 회사에서 몇 m 안의 식당을 모을지 (1000m ≈ 걸어서 12~15분)
CATEGORY_CODES = {"FD6": "음식점"}   # 카페도 넣으려면 "CE7": "카페" 추가

## 1. API 키 불러오기

키는 코드에 직접 쓰지 않습니다. Colab 왼쪽의 **열쇠 아이콘(보안 비밀)** 에 `KAKAO_REST_API_KEY` 라는 이름으로 넣어 두면 여기서 읽어옵니다.

회사 주소도 같은 곳에 `COMPANY_ADDRESS` 라는 이름으로 넣어 두세요. 상세 주소는 노트북 파일과 결과물에 남기지 않습니다.

In [ ]:
import os

def load_secret(name, prompt):
    """보안 비밀(Colab) → .env 파일 → 가려진 입력칸 순서로 값을 읽는다. 키·주소를 파일에 남기지 않기 위해서."""
    try:
        from google.colab import userdata   # Colab 보안 비밀
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    try:
        from dotenv import load_dotenv      # 내 컴퓨터에서 실행할 때: .env 파일
        load_dotenv()
    except ImportError:
        pass
    value = os.environ.get(name)
    if not value:
        from getpass import getpass         # 마지막 방법: 화면에 가려진 입력칸
        value = getpass(prompt)
    return value

API_KEY = load_secret("KAKAO_REST_API_KEY", "카카오 REST API 키: ")
COMPANY_ADDRESS = COMPANY_ADDRESS or load_secret("COMPANY_ADDRESS", "회사 주소(화면에 안 보임): ")
print("키 불러오기 성공" if API_KEY else "키가 없습니다")
print("회사 주소 불러오기 성공" if COMPANY_ADDRESS else "회사 주소가 없습니다")

## 2. 카카오 API에 물어보는 함수

- 카카오는 한 번 검색에 **최대 45곳**까지만 돌려줍니다.
- 그래서 회사 주변을 네모 칸으로 나누고, 칸 안에 45곳이 넘으면 그 칸을 다시 4등분해서 빠짐없이 모읍니다.

In [ ]:
import math, time
import requests

BASE = "https://dapi.kakao.com/v2/local"
session = requests.Session()

def kakao_get(path, params):
    r = session.get(f"{BASE}/{path}", headers={"Authorization": f"KakaoAK {API_KEY}"},
                    params=params, timeout=10)
    if r.status_code == 401:
        raise RuntimeError("키가 틀렸습니다. REST API 키인지 확인하세요.")
    if r.status_code == 403:
        raise RuntimeError("권한이 없습니다. 카카오 개발자 사이트에서 앱의 '카카오맵' 사용 설정을 ON으로 켜세요.")
    r.raise_for_status()
    time.sleep(0.1)   # 너무 빠르게 연속 요청하지 않기
    return r.json()

def geocode(address):
    docs = kakao_get("search/address.json", {"query": address})["documents"]
    if not docs:
        raise ValueError(f"주소를 찾지 못했습니다: {address}")
    return float(docs[0]["x"]), float(docs[0]["y"])   # 경도, 위도

def distance_m(x1, y1, x2, y2):
    # 두 좌표 사이의 거리(m)
    R = 6371000
    p1, p2 = math.radians(y1), math.radians(y2)
    dp, dl = p2 - p1, math.radians(x2 - x1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def search_rect(code, rect, depth=0):
    # rect = (왼쪽 경도, 아래 위도, 오른쪽 경도, 위 위도)
    params = {"category_group_code": code, "rect": ",".join(f"{v:.7f}" for v in rect), "size": 15}
    first = kakao_get("search/category.json", {**params, "page": 1})
    total = first["meta"]["total_count"]
    if total > 45 and depth < 6:
        x1, y1, x2, y2 = rect
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        found = []
        for sub in [(x1, y1, mx, my), (mx, y1, x2, my), (x1, my, mx, y2), (mx, my, x2, y2)]:
            found += search_rect(code, sub, depth + 1)
        return found
    docs = list(first["documents"])
    page = 1
    while not first["meta"]["is_end"] and page < 3:
        page += 1
        res = kakao_get("search/category.json", {**params, "page": page})
        docs += res["documents"]
        if res["meta"]["is_end"]:
            break
    return docs

## 3. 회사 좌표 찾고 주변 식당 모으기

In [ ]:
cx, cy = geocode(COMPANY_ADDRESS)
print("회사 좌표를 찾았어요 (위치 정보라 화면에 출력하지 않아요)")

# 반경을 감싸는 네모 칸 (위도 1도 ≈ 111km)
dy = RADIUS_M / 111000
dx = RADIUS_M / (111000 * math.cos(math.radians(cy)))
area = (cx - dx, cy - dy, cx + dx, cy + dy)

raw = []
for code, label in CATEGORY_CODES.items():
    docs = search_rect(code, area)
    print(f"{label}: {len(docs)}건 받음 (중복 포함)")
    raw += docs

## 4. 표로 정리하기

- 같은 식당이 두 칸에 걸쳐 두 번 나올 수 있어서 `id`로 중복을 지웁니다.
- 네모 칸의 모서리 쪽은 반경보다 멀어서 빼냅니다.
- 카카오 분류 `음식점 > 한식 > 육류,고기` 에서 두 번째 단계를 **메뉴 카테고리**로 씁니다. 고기·해산물은 저녁 카테고리에 맞춰 따로 뺍니다.

In [ ]:
import pandas as pd
from datetime import date

def menu_category(category_name):
    parts = [p.strip() for p in str(category_name).split(">")]
    if len(parts) < 2:
        return "기타"
    if "육류,고기" in parts:
        return "고기"
    if "해물,생선" in parts:
        return "횟집·해산물"
    return parts[1]

df = pd.DataFrame(raw)
n_before = len(df)
df = df.drop_duplicates(subset="id")
print(f"중복 제거: {n_before} → {len(df)}")

df["x"] = df["x"].astype(float)
df["y"] = df["y"].astype(float)
df["distance_m"] = [round(distance_m(cx, cy, x, y)) for x, y in zip(df["x"], df["y"])]
n_before = len(df)
df = df[df["distance_m"] <= RADIUS_M]
print(f"반경 {RADIUS_M}m 밖 제외: {n_before} → {len(df)}")

restaurants = pd.DataFrame({
    "restaurant_id": "K" + df["id"].astype(str),
    "name": df["place_name"],
    "menu_category": df["category_name"].map(menu_category),
    "kakao_category": df["category_name"],
    "distance_m": df["distance_m"],
    "walk_min": (df["distance_m"] / 67).round().astype(int),   # 분당 약 67m 걷는다고 가정
    "address": df["road_address_name"].where(df["road_address_name"] != "", df["address_name"]),
    "phone": df["phone"],
    "x": df["x"], "y": df["y"],
    "place_url": df["place_url"],
    "data_type": "실제",
    "collected_at": date.today().isoformat(),
}).sort_values("distance_m").reset_index(drop=True)

restaurants.head(10)

## 5. 검증: 결과가 말이 되는지 확인

In [ ]:
print("식당 수:", len(restaurants))
print("ID 중복:", restaurants["restaurant_id"].duplicated().sum(), "건 (0이어야 정상)")
print("메뉴 카테고리 빈칸:", restaurants["menu_category"].isna().sum(), "건")
print("가장 먼 식당:", restaurants["distance_m"].max(), "m (설정한 반경 이하여야 정상)")
print()
print("메뉴 카테고리별 식당 수")
restaurants["menu_category"].value_counts()

In [ ]:
# 직접 확인: 무작위 3곳의 카카오맵 링크를 열어 이름·위치가 맞는지 눈으로 봅니다
restaurants.sample(3, random_state=1)[["name", "menu_category", "distance_m", "place_url"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(restaurants["x"], restaurants["y"], s=8, alpha=0.6, label="restaurant")
ax.scatter([cx], [cy], s=120, marker="*", color="red", label="company")
ax.set_title(f"{len(restaurants)} restaurants within {RADIUS_M}m")  # Colab 기본 글꼴은 한글이 깨져서 영어로 씀
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend()
plt.show()

## 6. 저장하기

`restaurants.csv` 가 내 컴퓨터로 내려받아집니다. 이 파일은 깃허브에 올리지 않습니다(원본 데이터). 다음 노트북(01c)에서 이 파일을 씁니다.

In [ ]:
restaurants.to_csv("restaurants.csv", index=False, encoding="utf-8-sig")
print("저장 완료: restaurants.csv")
try:
    from google.colab import files
    files.download("restaurants.csv")
except ImportError:
    pass